# 03 - Feature Engineering

Loads raw data from Athena, engineers features, and stores them in SageMaker Feature Store.

Features created:
- `review_length` — character count of review text
- `word_count` — word count of review text
- `vader_score` — VADER compound sentiment score (-1 to 1)
- `sentiment` — binary target: 0 = negative (1-2 stars), 1 = positive (4-5 stars)

Note: 3-star reviews are dropped as ambiguous.

**Prerequisite:** Run `01_setup_athena.ipynb` first to create `project_config.json`.

## 0. Install Dependencies

In [1]:
import importlib, subprocess

def install_if_missing(package):
    if importlib.util.find_spec(package) is None:
        print(f'Installing {package}...')
        subprocess.run(['pip', 'install', package, '--quiet'], check=True)
        print(f'{package} installed')
    else:
        print(f'{package} already installed, skipping')

install_if_missing('nltk')

import nltk
nltk.download('vader_lexicon', quiet=True)
print('Dependencies ready')

nltk already installed, skipping


Dependencies ready


## 1. Setup

In [2]:
import boto3, json, time
import pandas as pd
from nltk.sentiment.vader import SentimentIntensityAnalyzer

with open('project_config.json') as f:
    cfg = json.load(f)

REGION         = cfg['REGION']
GLUE_DB        = cfg['GLUE_DB']
ATHENA_RESULTS = cfg['ATHENA_RESULTS']
SOURCE_BUCKET  = cfg['SOURCE_BUCKET']

session   = boto3.Session(region_name=REGION)
athena    = session.client('athena')
s3        = session.client('s3')
sm_client = session.client('sagemaker')

def get_role():
    try:
        import sagemaker
        return sagemaker.get_execution_role()
    except Exception:
        pass
    try:
        with open('/opt/ml/metadata/resource-metadata.json') as f:
            meta = json.load(f)
        return meta['ExecutionRoleArn']
    except Exception:
        pass
    try:
        iam = session.client('iam')
        return iam.get_role(RoleName='LabRole')['Role']['Arn']
    except Exception:
        pass
    raise RuntimeError('Could not detect IAM role — set MANUAL_ROLE_ARN manually')

MANUAL_ROLE_ARN = ''
role = MANUAL_ROLE_ARN if MANUAL_ROLE_ARN else get_role()

print(f'Region : {REGION}')
print(f'Role   : {role}')

Region : us-east-1
Role   : arn:aws:iam::476629097825:role/LabRole


## 2. Load Data from Athena

Samples 100k rows randomly from 2019 data.

In [3]:
def run_athena_query(sql):
    response = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={'Database': GLUE_DB},
        ResultConfiguration={'OutputLocation': ATHENA_RESULTS}
    )
    query_id = response['QueryExecutionId']
    while True:
        status = athena.get_query_execution(QueryExecutionId=query_id)
        state  = status['QueryExecution']['Status']['State']
        if state == 'SUCCEEDED':
            break
        elif state in ['FAILED', 'CANCELLED']:
            reason = status['QueryExecution']['Status']['StateChangeReason']
            raise Exception(f'Query {state}: {reason}')
        time.sleep(2)
    rows, next_token = [], None
    while True:
        kwargs = {'QueryExecutionId': query_id}
        if next_token:
            kwargs['NextToken'] = next_token
        page = athena.get_query_results(**kwargs)
        rows.extend(page['ResultSet']['Rows'])
        next_token = page.get('NextToken')
        if not next_token:
            break
    headers = [c['VarCharValue'] for c in rows[0]['Data']]
    data    = [[c.get('VarCharValue', '') for c in row['Data']] for row in rows[1:]]
    df = pd.DataFrame(data, columns=headers)
    # Fix dtypes — Athena returns everything as string
    df['stars']  = pd.to_numeric(df['stars'])
    df['useful'] = pd.to_numeric(df['useful'])
    df['funny']  = pd.to_numeric(df['funny'])
    df['cool']   = pd.to_numeric(df['cool'])
    df['date']   = pd.to_datetime(df['date'])
    return df

In [4]:
print('Loading data from Athena...')
df = run_athena_query(
    'SELECT * FROM yelp_reviews_2019 ORDER BY rand() LIMIT 100000'
)

print(f'Loaded {len(df):,} rows')
print(f'Star distribution:')
print(df['stars'].value_counts().sort_index())

Loading data from Athena...


Loaded 100,000 rows
Star distribution:
stars
1    17224
2     7164
3     8214
4    16251
5    51147
Name: count, dtype: int64


## 3. Create Binary Target

Drop 3-star reviews (ambiguous), then create binary sentiment label.

In [5]:
before = len(df)

df = df[df['stars'] != 3].copy()
df['sentiment'] = (df['stars'] >= 4).astype(int)

print(f'Dropped {before - len(df):,} three-star reviews')
print(f'Remaining rows: {len(df):,}')
print(f'\nClass distribution:')
print(df['sentiment'].value_counts())
print(f'\nPositive rate: {df["sentiment"].mean()*100:.1f}%')

Dropped 8,214 three-star reviews
Remaining rows: 91,786

Class distribution:
sentiment
1    67398
0    24388
Name: count, dtype: int64

Positive rate: 73.4%


## 4. Text Features

In [6]:
df['review_length'] = df['text'].str.len()
df['word_count']    = df['text'].str.split().str.len()

print('review_length stats:')
print(df['review_length'].describe())
print('\nword_count stats:')
print(df['word_count'].describe())

review_length stats:
count    91786.000000
mean       512.867289
std        479.486755
min         13.000000
25%        212.000000
50%        365.000000
75%        643.000000
max       5000.000000
Name: review_length, dtype: float64

word_count stats:
count    91786.000000
mean        94.716406
std         89.432561
min          1.000000
25%         38.000000
50%         67.000000
75%        119.000000
max       1004.000000
Name: word_count, dtype: float64


## 5. VADER Sentiment Score

Computes compound sentiment score (-1 = most negative, +1 = most positive).
This may take 2-3 minutes for 100k rows.

In [7]:
sid = SentimentIntensityAnalyzer()

print('Computing VADER scores...')
start = time.time()

df['vader_score'] = df['text'].apply(
    lambda x: sid.polarity_scores(x)['compound']
)

print(f'Done in {time.time() - start:.0f}s')
print(f'\nvader_score stats:')
print(df['vader_score'].describe())
print(f'\nAverage VADER score by sentiment:')
print(df.groupby('sentiment')['vader_score'].mean())

Computing VADER scores...


Done in 69s

vader_score stats:
count    91786.000000
mean         0.613728
std          0.593307
min         -0.998700
25%          0.616325
50%          0.914200
75%          0.968200
max          0.999800
Name: vader_score, dtype: float64

Average VADER score by sentiment:
sentiment
0   -0.103002
1    0.873078
Name: vader_score, dtype: float64


## 6. Feature Summary

In [8]:
feature_cols = ['review_length', 'word_count', 'useful', 'funny', 'cool', 'vader_score']
target_col   = 'sentiment'

print('Feature matrix shape:', df[feature_cols].shape)
print()
print('Features:')
print(df[feature_cols].describe())
print()
print('Target distribution:')
print(df[target_col].value_counts())

Feature matrix shape: (91786, 6)

Features:
       review_length    word_count        useful         funny          cool  \
count   91786.000000  91786.000000  91786.000000  91786.000000  91786.000000   
mean      512.867289     94.716406      0.917624      0.216275      0.460441   
std       479.486755     89.432561      2.636943      1.291149      2.135852   
min        13.000000      1.000000      0.000000      0.000000      0.000000   
25%       212.000000     38.000000      0.000000      0.000000      0.000000   
50%       365.000000     67.000000      0.000000      0.000000      0.000000   
75%       643.000000    119.000000      1.000000      0.000000      0.000000   
max      5000.000000   1004.000000    176.000000    126.000000    172.000000   

        vader_score  
count  91786.000000  
mean       0.613728  
std        0.593307  
min       -0.998700  
25%        0.616325  
50%        0.914200  
75%        0.968200  
max        0.999800  

Target distribution:
sentiment
1    

## 7. Save Features to S3

In [9]:
df_features = df[[
    'review_id',
    'review_length',
    'word_count',
    'useful',
    'funny',
    'cool',
    'vader_score',
    'sentiment'
]].copy()

local_path = '/tmp/yelp_features.parquet'
s3_key     = 'features/yelp_features.parquet'

df_features.to_parquet(local_path, index=False)
s3.upload_file(local_path, SOURCE_BUCKET, s3_key)

print(f'Features saved to s3://{SOURCE_BUCKET}/{s3_key}')
print(f'Rows    : {len(df_features):,}')
print(f'Columns : {list(df_features.columns)}')

Features saved to s3://aai540-group1-yelp-data/features/yelp_features.parquet
Rows    : 91,786
Columns : ['review_id', 'review_length', 'word_count', 'useful', 'funny', 'cool', 'vader_score', 'sentiment']


## 8. SageMaker Feature Store

In [10]:
FEATURE_GROUP_NAME = 'yelp-review-features'

try:
    sm_client.create_feature_group(
        FeatureGroupName=FEATURE_GROUP_NAME,
        RecordIdentifierFeatureName='review_id',
        EventTimeFeatureName='event_time',
        FeatureDefinitions=[
            {'FeatureName': 'review_id',     'FeatureType': 'String'},
            {'FeatureName': 'review_length', 'FeatureType': 'Integral'},
            {'FeatureName': 'word_count',    'FeatureType': 'Integral'},
            {'FeatureName': 'useful',        'FeatureType': 'Integral'},
            {'FeatureName': 'funny',         'FeatureType': 'Integral'},
            {'FeatureName': 'cool',          'FeatureType': 'Integral'},
            {'FeatureName': 'vader_score',   'FeatureType': 'Fractional'},
            {'FeatureName': 'sentiment',     'FeatureType': 'Integral'},
            {'FeatureName': 'event_time',    'FeatureType': 'Fractional'},
        ],
        OnlineStoreConfig={'EnableOnlineStore': True},
        OfflineStoreConfig={
            'S3StorageConfig': {
                'S3Uri': f's3://{SOURCE_BUCKET}/feature-store/'
            }
        },
        RoleArn=role
    )
    print(f'Feature group {FEATURE_GROUP_NAME} created')
except sm_client.exceptions.ResourceInUse:
    print(f'Feature group {FEATURE_GROUP_NAME} already exists, skipping')

# Wait for ready
print('Waiting for feature group...')
while True:
    status = sm_client.describe_feature_group(
        FeatureGroupName=FEATURE_GROUP_NAME
    )['FeatureGroupStatus']
    if status == 'Created':
        print('Feature group ready')
        break
    elif status == 'CreateFailed':
        raise Exception('Feature group creation failed')
    time.sleep(5)

Feature group yelp-review-features already exists, skipping
Waiting for feature group...


Feature group ready


### 8.1 Ingest Records into the Feature Store

Creating the feature group registers the schema but leaves the store empty.
This step ingests the engineered feature records into the online and offline
stores so the feature group is queryable in the SageMaker console and can serve
features to downstream jobs, then reads one record back to confirm ingestion.

In [11]:
from concurrent.futures import ThreadPoolExecutor

fs_runtime = session.client('sagemaker-featurestore-runtime')

# Creating the feature group only registers the schema. To make the store
# usable (and queryable from the console / offline store), we ingest the
# engineered records. Feature Store needs an event_time per record and takes
# every value as a string; integral features must be whole numbers.
event_time    = time.time()
integral_cols = ['review_length', 'word_count', 'useful', 'funny', 'cool', 'sentiment']

ingest_df = df_features.copy()
ingest_df['event_time'] = event_time
for col in integral_cols:
    ingest_df[col] = ingest_df[col].astype('int64')

# Online put_record is one API call per record, so ingest a representative
# sample for the demo. The full feature set lives in the offline store / S3 parquet.
INGEST_SAMPLE_SIZE = 2000
sample_df = ingest_df.head(INGEST_SAMPLE_SIZE)


def put_record(row):
    record = [
        {'FeatureName': 'review_id',     'ValueAsString': str(row['review_id'])},
        {'FeatureName': 'review_length', 'ValueAsString': str(int(row['review_length']))},
        {'FeatureName': 'word_count',    'ValueAsString': str(int(row['word_count']))},
        {'FeatureName': 'useful',        'ValueAsString': str(int(row['useful']))},
        {'FeatureName': 'funny',         'ValueAsString': str(int(row['funny']))},
        {'FeatureName': 'cool',          'ValueAsString': str(int(row['cool']))},
        {'FeatureName': 'vader_score',   'ValueAsString': str(float(row['vader_score']))},
        {'FeatureName': 'sentiment',     'ValueAsString': str(int(row['sentiment']))},
        {'FeatureName': 'event_time',    'ValueAsString': str(float(row['event_time']))},
    ]
    fs_runtime.put_record(FeatureGroupName=FEATURE_GROUP_NAME, Record=record)


print(f'Ingesting {len(sample_df):,} records into {FEATURE_GROUP_NAME}...')
start = time.time()
with ThreadPoolExecutor(max_workers=10) as pool:
    list(pool.map(put_record, [r for _, r in sample_df.iterrows()]))
print(f'Ingested {len(sample_df):,} records in {time.time() - start:.0f}s')

# Verify: read one record back from the online store
sample_id = str(sample_df.iloc[0]['review_id'])
resp = fs_runtime.get_record(
    FeatureGroupName=FEATURE_GROUP_NAME,
    RecordIdentifierValueAsString=sample_id,
)
print(f'\nRetrieved record for review_id={sample_id}:')
for feat in resp['Record']:
    print(f"  {feat['FeatureName']:<14} = {feat['ValueAsString']}")

Ingesting 2,000 records into yelp-review-features...


Ingested 2,000 records in 4s

Retrieved record for review_id=tYnSLrWfG6L0Vp5p5zQlLw:
  review_id      = tYnSLrWfG6L0Vp5p5zQlLw
  review_length  = 526
  word_count     = 93
  useful         = 0
  funny          = 0
  cool           = 0
  vader_score    = 0.4778
  sentiment      = 0
  event_time     = 1781496228.9288774


## 9. Update Config for Next Notebooks

In [12]:
cfg['FEATURE_GROUP_NAME'] = FEATURE_GROUP_NAME
cfg['FEATURES_S3_PATH']   = f's3://{SOURCE_BUCKET}/features/yelp_features.parquet'
cfg['FEATURE_COLS']       = feature_cols
cfg['TARGET_COL']         = target_col

with open('project_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

print('Config updated')
print(json.dumps(cfg, indent=2))

Config updated
{
  "REGION": "us-east-1",
  "SOURCE_BUCKET": "aai540-group1-yelp-data",
  "GLUE_DB": "yelp_reviews_db",
  "ATHENA_RESULTS": "s3://aai540-group1-yelp-data/athena-results/qmou/",
  "TABLES": [
    "yelp_reviews_2019",
    "yelp_reviews_2020_2022"
  ],
  "FEATURE_GROUP_NAME": "yelp-review-features",
  "FEATURES_S3_PATH": "s3://aai540-group1-yelp-data/features/yelp_features.parquet",
  "FEATURE_COLS": [
    "review_length",
    "word_count",
    "useful",
    "funny",
    "cool",
    "vader_score"
  ],
  "TARGET_COL": "sentiment"
}
